In [ ]:
import os
import json
import pandas as pd
from datetime import datetime
import logging
from add_ids_to_dataset_utils import *
from configs import AddIdsToDatasetConfig

In [ ]:
file_ext = os.path.splitext(AddIdsToDatasetConfig.RAW_DATASET)[1].lower()

try:
    if file_ext == '.csv':
        dataset = pd.read_csv(AddIdsToDatasetConfig.RAW_DATASET, encoding='utf-8-sig') 
    elif file_ext in ['.xlsx', '.xls']:
        dataset = pd.read_excel(AddIdsToDatasetConfig.RAW_DATASET)
except Exception as e:
    print(f"读取文件出错: {e}")

ids_map_table = pd.read_excel(AddIdsToDatasetConfig.IDS_MAP_TABLE)

In [ ]:
component_columns = ['c1', 'c2', 'c3']
key_column = 'compound'
dataset_with_ids = add_ids_to_dataset(dataset, ids_map_table, component_columns, key_column)

In [ ]:
dataset_with_ids_sum_output_path = os.path.join(AddIdsToDatasetConfig.MISSING_DATA_MAG_DIR, 'ds-with-ids_qlty-sum.xlsx')
data_quality_summary(dataset_with_ids, dataset_with_ids_sum_output_path)

In [ ]:
# 获取所有不包含'3'的列名
columns_without_3 = [col for col in dataset_with_ids.columns if '3' not in col]
ismiles_columns = [col for col in columns_without_3 if 'ismiles' in col]

# 1. 记录原始行数
original_count = len(dataset_with_ids)

# 只对这些列进行dropna操作
dataset_with_ids_cleaned = dataset_with_ids.dropna(subset=ismiles_columns)

# 2. 记录清理后的行数，并计算差值
cleaned_count = len(dataset_with_ids_cleaned)
dropped_count = original_count - cleaned_count

# 3. 打印结果
print(f"缺失值清理完成：")
print(f"  - 原始数据共 {original_count} 行")
print(f"  - 删除了 {dropped_count} 行含有空值的数据")
print(f"  - 剩余有效数据 {cleaned_count} 行")

# 保存文件
dataset_with_ids_cleaned.to_excel(AddIdsToDatasetConfig.DATASET_WITH_IDS, index=False)